In [1]:
!apt-get update -y
!apt-get install -y ffmpeg --quiet
!pip install -q --no-cache-dir gTTS

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,695 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/u

In [2]:
from google.colab import drive
from pathlib import Path
import subprocess, json
from gtts import gTTS
from IPython.display import Video, display

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/kride-track-b")
PHOTO_DIR = ROOT / "travel_photo_inputs"

COG_DIR = ROOT / "cogvideo_photo_outputs"
TTS_DIR = ROOT / "cogvideo_photo_tts"
FINAL_DIR = ROOT / "cogvideo_photo_final"

COG_DIR.mkdir(parents=True, exist_ok=True)
TTS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

travel_cases = [
    {
        "id": "gangneung_beach",
        "place": "강릉 해변",
        "image": PHOTO_DIR / "gangneung_beach.jpg",
        "prompt": "A cinematic AI-generated travel video from a real photo of Gangneung beach, gentle waves, bright daylight, smooth camera movement.",
        "tts": "강릉 해변 사진을 바탕으로 생성형 영상 브랜치를 구성합니다. 바다와 하늘의 분위기를 자연스럽게 살려봅니다.",
        "bgm": "bright_travel",
    },
    {
        "id": "gwanghwamun",
        "place": "광화문",
        "image": PHOTO_DIR / "gwanghwamun.jpg",
        "prompt": "A cinematic AI-generated travel video from a real photo of Gwanghwamun in Seoul, historical gate, slow forward camera movement.",
        "tts": "광화문 사진을 바탕으로 생성형 여행 영상 케이스를 구성합니다. 역사적인 건축과 도심의 분위기를 함께 표현합니다.",
        "bgm": "cinematic_memory",
    },
    {
        "id": "gyeongbokgung",
        "place": "경복궁",
        "image": PHOTO_DIR / "gyeongbokgung.jpg",
        "prompt": "A cinematic AI-generated travel video from a real photo of Gyeongbokgung palace, traditional Korean architecture, warm light, elegant motion.",
        "tts": "경복궁 사진을 기반으로 전통적인 공간감을 살린 생성형 영상 브랜치를 구성합니다.",
        "bgm": "cinematic_memory",
    },
]

bgm_map = {
    "bright_travel": ROOT / "bgm_bright_travel.wav",
    "cute_character": ROOT / "bgm_cute_character.wav",
    "city_walk": ROOT / "bgm_city_walk.wav",
    "cinematic_memory": ROOT / "bgm_cinematic_memory.wav",
}

def build_cogvideo_fallback_filter(case_id, duration=7, fps=8):
    frames = duration * fps

    if "beach" in case_id:
        return (
            "scale=768:-1,"
            f"zoompan=z='min(zoom+0.0012,1.08)':"
            "x='iw/2-(iw/zoom/2)':"
            "y='ih/2-(ih/zoom/2)':"
            f"d={frames}:s=512x320:fps={fps},"
            "eq=saturation=1.08:contrast=1.03,"
            "format=yuv420p"
        )

    if "gwanghwamun" in case_id:
        return (
            "scale=768:-1,"
            f"zoompan=z='1.06':"
            f"x='(iw-iw/zoom)*on/{frames}':"
            "y='ih/2-(ih/zoom/2)':"
            f"d={frames}:s=512x320:fps={fps},"
            "eq=saturation=1.02:contrast=1.05,"
            "format=yuv420p"
        )

    return (
        "scale=768:-1,"
        f"zoompan=z='1.02+0.04*sin(on/12)':"
        "x='iw/2-(iw/zoom/2)':"
        "y='ih/2-(ih/zoom/2)':"
        f"d={frames}:s=512x320:fps={fps},"
        "eq=saturation=1.03:contrast=1.04,"
        "format=yuv420p"
    )

final_cog_outputs = []

for case in travel_cases:
    image = Path(case["image"])
    if not image.exists():
        print("⚠️ 사진 없음, 건너뜀:", image)
        continue

    fallback_video = COG_DIR / f"{case['id']}_cogvideo_photo_fallback.mp4"
    route_json = COG_DIR / f"{case['id']}_cogvideo_route.json"

    record = {
        "id": case["id"],
        "place": case["place"],
        "route": "cogvideox",
        "input_type": "real_travel_photo",
        "image": str(image),
        "prompt": case["prompt"],
        "model_attempted": "THUDM/CogVideoX-2b",
        "status": "fallback_used",
        "fallback_type": "photo_motion_video",
        "fallback_reason": "CogVideoX-2b crashed or was unstable on Colab Tesla T4. This output preserves the CogVideoX branch using a real travel photo.",
        "fallback_video": str(fallback_video),
    }
    route_json.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")

    if not fallback_video.exists():
        print("▶️ CogVideoX fallback 생성:", case["place"])
        vf = build_cogvideo_fallback_filter(case["id"])
        subprocess.run([
            "ffmpeg", "-y",
            "-loop", "1",
            "-i", str(image),
            "-vf", vf,
            "-t", "7",
            "-c:v", "libx264",
            "-preset", "veryfast",
            "-crf", "23",
            str(fallback_video),
            "-loglevel", "quiet",
        ], check=True)
    else:
        print("✅ 기존 fallback 사용:", fallback_video.name)

    tts_mp3 = TTS_DIR / f"{case['id']}.mp3"
    tts_wav = TTS_DIR / f"{case['id']}.wav"
    final = FINAL_DIR / f"{case['id']}_cogvideo_photo_final.mp4"
    bgm = bgm_map[case["bgm"]]

    assert bgm.exists(), f"BGM 없음: {bgm}"

    if not tts_wav.exists():
        print("▶️ TTS 생성:", case["place"])
        gTTS(case["tts"], lang="ko").save(str(tts_mp3))
        subprocess.run([
            "ffmpeg", "-y",
            "-i", str(tts_mp3),
            "-ar", "44100",
            "-ac", "2",
            str(tts_wav),
            "-loglevel", "quiet",
        ], check=True)

    if not final.exists():
        print("▶️ 최종 합성:", case["place"])
        subprocess.run([
            "ffmpeg", "-y",
            "-i", str(fallback_video),
            "-i", str(tts_wav),
            "-stream_loop", "-1",
            "-i", str(bgm),
            "-filter_complex",
            "[2:a]volume=-18dB[bgm];[1:a][bgm]amix=inputs=2:duration=first:dropout_transition=0:normalize=0[aout]",
            "-map", "0:v",
            "-map", "[aout]",
            "-c:v", "copy",
            "-c:a", "aac",
            "-shortest",
            str(final),
            "-loglevel", "quiet",
        ], check=True)

    final_cog_outputs.append(final)
    print("final:", final.name, final.exists())

print("\n✅ CogVideoX real-photo fallback final outputs:")
for p in final_cog_outputs:
    print(p.name, p.exists())

if final_cog_outputs:
    display(Video(str(final_cog_outputs[0]), embed=True, width=640))

Mounted at /content/drive
▶️ CogVideoX fallback 생성: 강릉 해변
▶️ TTS 생성: 강릉 해변
▶️ 최종 합성: 강릉 해변
final: gangneung_beach_cogvideo_photo_final.mp4 True
▶️ CogVideoX fallback 생성: 광화문
▶️ TTS 생성: 광화문
▶️ 최종 합성: 광화문
final: gwanghwamun_cogvideo_photo_final.mp4 True
▶️ CogVideoX fallback 생성: 경복궁
▶️ TTS 생성: 경복궁
▶️ 최종 합성: 경복궁
final: gyeongbokgung_cogvideo_photo_final.mp4 True

✅ CogVideoX real-photo fallback final outputs:
gangneung_beach_cogvideo_photo_final.mp4 True
gwanghwamun_cogvideo_photo_final.mp4 True
gyeongbokgung_cogvideo_photo_final.mp4 True


실제 여행사진 입력
-> route = cogvideox
-> CogVideoX 시도 기록
-> T4에서 실패/불안정하면 photo_motion fallback 생성
-> DagsHub에 "CogVideoX 브랜치의 실제 여행사진 fallback"으로 기록

In [3]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/kride-track-b")

for d in [
    ROOT / "cogvideo_photo_outputs",
    ROOT / "cogvideo_photo_tts",
    ROOT / "cogvideo_photo_final",
]:
    print("\nDIR:", d, d.exists())
    if d.exists():
        for p in sorted(d.glob("*")):
            print(" -", p.name, round(p.stat().st_size / 1024 / 1024, 2), "MB")


DIR: /content/drive/MyDrive/kride-track-b/cogvideo_photo_outputs True
 - gangneung_beach_cogvideo_photo_fallback.mp4 0.08 MB
 - gangneung_beach_cogvideo_route.json 0.0 MB
 - gwanghwamun_cogvideo_photo_fallback.mp4 0.06 MB
 - gwanghwamun_cogvideo_route.json 0.0 MB
 - gyeongbokgung_cogvideo_photo_fallback.mp4 0.12 MB
 - gyeongbokgung_cogvideo_route.json 0.0 MB

DIR: /content/drive/MyDrive/kride-track-b/cogvideo_photo_tts True
 - gangneung_beach.mp3 0.07 MB
 - gangneung_beach.wav 1.65 MB
 - gwanghwamun.mp3 0.08 MB
 - gwanghwamun.wav 1.75 MB
 - gyeongbokgung.mp3 0.06 MB
 - gyeongbokgung.wav 1.29 MB

DIR: /content/drive/MyDrive/kride-track-b/cogvideo_photo_final True
 - gangneung_beach_cogvideo_photo_final.mp4 0.18 MB
 - gwanghwamun_cogvideo_photo_final.mp4 0.17 MB
 - gyeongbokgung_cogvideo_photo_final.mp4 0.23 MB


In [1]:
 from google.colab import drive, userdata
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/kride-track-b")

COG_DIR = ROOT / "cogvideo_photo_outputs"
TTS_DIR = ROOT / "cogvideo_photo_tts"
FINAL_DIR = ROOT / "cogvideo_photo_final"

for d in [COG_DIR, TTS_DIR, FINAL_DIR]:
    print("\nDIR:", d, d.exists())
    if d.exists():
        for p in sorted(d.glob("*")):
            print(" -", p.name, round(p.stat().st_size / 1024 / 1024, 2), "MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

DIR: /content/drive/MyDrive/kride-track-b/cogvideo_photo_outputs True
 - gangneung_beach_cogvideo_photo_fallback.mp4 0.08 MB
 - gangneung_beach_cogvideo_route.json 0.0 MB
 - gwanghwamun_cogvideo_photo_fallback.mp4 0.06 MB
 - gwanghwamun_cogvideo_route.json 0.0 MB
 - gyeongbokgung_cogvideo_photo_fallback.mp4 0.12 MB
 - gyeongbokgung_cogvideo_route.json 0.0 MB

DIR: /content/drive/MyDrive/kride-track-b/cogvideo_photo_tts True
 - gangneung_beach.mp3 0.07 MB
 - gangneung_beach.wav 1.65 MB
 - gwanghwamun.mp3 0.08 MB
 - gwanghwamun.wav 1.75 MB
 - gyeongbokgung.mp3 0.06 MB
 - gyeongbokgung.wav 1.29 MB

DIR: /content/drive/MyDrive/kride-track-b/cogvideo_photo_final True
 - gangneung_beach_cogvideo_photo_final.mp4 0.18 MB
 - gwanghwamun_cogvideo_photo_final.mp4 0.17 MB
 - gyeongbokgung_cogvideo_photo_final.mp4 0.23 MB


In [2]:
!pip install -q --no-cache-dir --force-reinstall numpy==1.26.4 pandas==2.1.4
!pip install -q --no-cache-dir mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 199.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 151.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 kB 292.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 295.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.1.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have num

In [3]:
import numpy, pandas, mlflow
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("mlflow:", mlflow.__version__)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

1. 3D Photo Light
   실제 여행사진 -> pan/zoom/dolly motion
   성공한 정식 경량 브랜치

2. CogVideoX
   실제 여행사진 -> CogVideoX 시도
   T4에서 세션 다운
   현재 결과는 fallback, 진짜 CogVideoX 결과 아님

3. AnimatedDrawings
   캐릭터/낙서 계열 -> 순차 애니메이션
   성공

4. TTS/BGM/FFmpeg
   segment sync 최종 합성
   성공

In [4]:
import os, json, mlflow
from pathlib import Path
from google.colab import userdata

ROOT = Path("/content/drive/MyDrive/kride-track-b")

COG_DIR = ROOT / "cogvideo_photo_outputs"
TTS_DIR = ROOT / "cogvideo_photo_tts"
FINAL_DIR = ROOT / "cogvideo_photo_final"

route_jsons = sorted(COG_DIR.glob("*_cogvideo_route.json"))
fallback_videos = sorted(COG_DIR.glob("*_cogvideo_photo_fallback.mp4"))
tts_outputs = sorted(TTS_DIR.glob("*.wav"))
final_outputs = sorted(FINAL_DIR.glob("*_cogvideo_photo_final.mp4"))

DAGSHUB_USERNAME = "myelin24m"
DAGSHUB_REPO = "Kride"
DAGSHUB_TOKEN = userdata.get("DAGSHUB_TOKEN")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("track-b-colab-cogvideox-real-photo")

with mlflow.start_run(run_name="cogvideox-real-photo-fallback-not-model-output") as run:
    mlflow.log_param("route", "cogvideox")
    mlflow.log_param("input_type", "real_travel_photo")
    mlflow.log_param("model_attempted", "THUDM/CogVideoX-2b")
    mlflow.log_param("actual_model_executed", False)
    mlflow.log_param("status", "fallback_used")
    mlflow.log_param("fallback_type", "photo_motion_video")
    mlflow.log_param("case_count", len(final_outputs))
    mlflow.log_param(
        "note",
        "These outputs are real-travel-photo fallback videos for the CogVideoX branch. CogVideoX was not executed successfully in this run because it crashed/was unstable on Colab Tesla T4."
    )

    for p in route_jsons:
        mlflow.log_artifact(str(p), artifact_path="route_metadata")

    for p in fallback_videos:
        mlflow.log_artifact(str(p), artifact_path="cogvideo_photo_fallback")

    for p in tts_outputs:
        mlflow.log_artifact(str(p), artifact_path="tts")

    for p in final_outputs:
        mlflow.log_metric(f"{p.stem}_size_mb", p.stat().st_size / 1024 / 1024)
        mlflow.log_artifact(str(p), artifact_path="final_cogvideo_photo_fallback")

    summary = {
        "route": "cogvideox",
        "input_type": "real_travel_photo",
        "model_attempted": "THUDM/CogVideoX-2b",
        "actual_model_executed": False,
        "status": "fallback_used",
        "fallback_type": "photo_motion_video",
        "route_jsons": [str(p) for p in route_jsons],
        "fallback_videos": [str(p) for p in fallback_videos],
        "tts_outputs": [str(p) for p in tts_outputs],
        "final_outputs": [str(p) for p in final_outputs],
    }

    summary_path = ROOT / "cogvideo_real_photo_fallback_summary.json"
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    mlflow.log_artifact(str(summary_path), artifact_path="summary")

    print("✅ CogVideoX real-photo fallback DagsHub 기록 완료")
    print("Run ID:", run.info.run_id)
    print("MLflow URI:", mlflow.get_tracking_uri())

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject